**spaCy**

In [77]:
import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)


Project root added: c:\Users\Samyak.Bora\Desktop\Training\Assessments_And_Projects\Group_Project\Final


In [ ]:
# !pip install spacy
# !python -m spacy download en_core_web_sm

In [54]:
import spacy
from spacy.util import minibatch, compounding
from spacy.training.example import Example

import pandas as pd
import random
import os

# from src.nlp.preprocess_text import clean_text
import json



In [ ]:
# Load Cleaned Dataset
df = pd.read_csv("..\\data\\subscription_dataset_cleaned.csv")
# src/nlp/preprocess_text.py/clean_text()
df["CleanedDescription"] = df["Description"].apply(clean_text)
df.head()


,CustomerID,TransactionID,Date,Description,Merchant,Amount,Balance,SubscriptionFlag,Month,DayOfWeek,TransactionType,Category,CleanedDescription
0,C1,T1,2025-01-01,Payment to Cafe Coffee Day,Cafe Coffee Day,-1889,6136,0,2025-01,Wednesday,Debit,Food,payment to cafe coffee day
1,C1,T2,2025-01-01,Payment to Flipkart,Flipkart,-955,5181,0,2025-01,Wednesday,Debit,Shopping,payment to flipkart
2,C1,T3,2025-01-02,Salary Credit,Employer,49954,55135,0,2025-01,Thursday,Credit,Others,salary credit
3,C1,T4,2025-01-02,Payment to Reliance Smart,Reliance Smart,-1582,53553,0,2025-01,Thursday,Debit,Shopping,payment to reliance smart
4,C1,T5,2025-01-02,Payment to Petrol Pump,Petrol Pump,-1257,52296,0,2025-01,Thursday,Debit,Fuel,payment to petrol pump


**Text Cleaning Function**

In [57]:
# # Text Cleaning Function
# def clean_text(text: str) -> str:
#     """Basic text cleaning:
#     - Lowercase
#     - Remove punctuation
#     - Remove digits
#     - Remove extra spaces
#     """
#     if not isinstance(text, str):
#         return ""

#     text = text.lower()
#     text = text.translate(str.maketrans("", "", string.punctuation))
#     text = re.sub(r"\d+", "", text)
#     text = re.sub(r"\s+", " ", text).strip()

#     return text

# df["CleanedDescription"] = df["Description"].apply(clean_text)
# df.head()

In [49]:
from sklearn.model_selection import train_test_split

# Cleaned text already exists: df["CleanedDescription"]
X = df["CleanedDescription"].tolist()
y = df["SubscriptionFlag"].tolist()

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

len(X_train), len(X_test)

(79356, 19840)

**Convert Dataset to spaCy Training Format**

In [59]:
# Convert Dataset into spaCy Training Format
train_data = []

for _, row in df.iterrows():
    text = row["CleanedDescription"]
    label = row["SubscriptionFlag"]  # from your dataset

    train_data.append(
        (text, {"cats": {"SUBSCRIPTION": float(label)}})
    )

# len(train_data), train_data[:3]

print("Length:",len(train_data))
train_data

Length: 99196


[('payment to cafe coffee day', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to flipkart', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('salary credit', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to reliance smart', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to petrol pump', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('electricity bill subscription', {'cats': {'SUBSCRIPTION': 1.0}}),
 ('payment to petrol pump', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to petrol pump', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to uber', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to swiggy', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to medical store', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to medical store', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to cafe coffee day', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to medical store', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to swiggy', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to petrol pump', {'cats': {'SUBSCRIPTION': 0.0}}),
 ('payment to

**Create spaCy Model**

In [60]:
# Create spaCy Model & Add TextCategorizer
# Create a blank English model
nlp = spacy.blank("en")

# Add multilabel text classifier (important) {instead of textcat}
textcat = nlp.add_pipe("textcat_multilabel")

# Add SINGLE label because this is binary classification
textcat.add_label("SUBSCRIPTION")

1

Train Model

In [61]:
n_iter = 12
optimizer = nlp.initialize()

for epoch in range(n_iter):
    random.shuffle(train_data)
    losses = {}

    batches = minibatch(train_data, size=compounding(4.0, 32.0, 1.5))

    for batch in batches:
        examples = []
        for text, annotations in batch:
            doc = nlp.make_doc(text)
            examples.append(Example.from_dict(doc, annotations))

        nlp.update(examples, sgd=optimizer, losses=losses)

    print(f"Epoch {epoch + 1}/{n_iter}, Loss: {losses}")


Epoch 1/12, Loss: {'textcat_multilabel': 0.0827654837221564}
Epoch 2/12, Loss: {'textcat_multilabel': 3.6292678647412407e-07}
Epoch 3/12, Loss: {'textcat_multilabel': 1.3715013341653818e-07}
Epoch 4/12, Loss: {'textcat_multilabel': 5.695304311313784e-08}
Epoch 5/12, Loss: {'textcat_multilabel': 2.727415790069153e-08}
Epoch 6/12, Loss: {'textcat_multilabel': 1.2429194346918332e-08}
Epoch 7/12, Loss: {'textcat_multilabel': 6.193748370292968e-09}
Epoch 8/12, Loss: {'textcat_multilabel': 3.409182593386707e-09}
Epoch 9/12, Loss: {'textcat_multilabel': 2.0214183941012656e-09}
Epoch 10/12, Loss: {'textcat_multilabel': 1.386352522208666e-09}
Epoch 11/12, Loss: {'textcat_multilabel': 1.182787613680889e-09}
Epoch 12/12, Loss: {'textcat_multilabel': 1.134827397363588e-09}


**Test & Create Clean Prediction Column**

In [63]:
predicted_flags = []

for desc in df["CleanedDescription"]:
    doc = nlp(desc)
    score = doc.cats["SUBSCRIPTION"]
    predicted_flags.append(1 if score >= 0.5 else 0)

df["PredictedSubscription"] = predicted_flags
df.head()

,CustomerID,TransactionID,Date,Description,Merchant,Amount,Balance,SubscriptionFlag,Month,DayOfWeek,TransactionType,Category,CleanedDescription,PredictedSubscription
0,C1,T1,2025-01-01,Payment to Cafe Coffee Day,Cafe Coffee Day,-1889,6136,0,2025-01,Wednesday,Debit,Food,payment to cafe coffee day,0
1,C1,T2,2025-01-01,Payment to Flipkart,Flipkart,-955,5181,0,2025-01,Wednesday,Debit,Shopping,payment to flipkart,0
2,C1,T3,2025-01-02,Salary Credit,Employer,49954,55135,0,2025-01,Thursday,Credit,Others,salary credit,0
3,C1,T4,2025-01-02,Payment to Reliance Smart,Reliance Smart,-1582,53553,0,2025-01,Thursday,Debit,Shopping,payment to reliance smart,0
4,C1,T5,2025-01-02,Payment to Petrol Pump,Petrol Pump,-1257,52296,0,2025-01,Thursday,Debit,Fuel,payment to petrol pump,0


In [75]:
# !pip install src
import sys
import os

# Get the project root (folder containing 'src')
project_root = os.path.abspath("..")   # if notebook is inside 'notebooks'
# project_root = os.path.abspath(".")  # if notebook is at project root

# Add it to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)

Project root added: c:\Users\Samyak.Bora\Desktop\Training\Assessments_And_Projects\Group_Project\Final


In [80]:
# FINAL STEP OF FR3: Generate subscription detection CSV for FR4

# from src.nlp.subscription_classifier import detect_subscription
from src.nlp.preprocess_text import clean_text

output_rows = []

for idx, row in df.iterrows():
    result = detect_subscription([row["Description"], row["Merchant"]])

    output_rows.append({
        "CustomerID": row["CustomerID"],
        "TransactionID": row["TransactionID"],
        "Date": row["Date"],
        "Description": row["Description"],
        "CleanedDescription": clean_text(row["Description"]),
        "Merchant": row["Merchant"],
        "Amount": row["Amount"],
        "Balance": row["Balance"],

        # True label
        "SubscriptionFlag": row["SubscriptionFlag"],

        # NLP prediction (0/1)
        "PredictedSubscription": 1 if result["TransactionType"] == "Subscription" else 0,

        # ML confidence score
        "Score": result["Score"],

        # For debugging and FR4
        "DetectionDetails": str(result)
    })

nlp_out_df = pd.DataFrame(output_rows)

nlp_out_df.to_csv("data/spacy_detected_subscriptions.csv", index=False)
print("🎉 File generated: data/spacy_detected_subscriptions.csv")
nlp_out_df.head()

Preprocess_text Performed ✅


KeyError: 'TransactionType'

**Save Model**

In [ ]:

nlp.to_disk("src/models/saved_models/spacy_subscription_model")
print("Saved spaCy model.")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'src\\models\\saved_models\\spacy_subscription_model'

**Save Final CSV(for Step 5)**

In [ ]:
df.to_csv("data/spacy_detected_subscriptions.csv", index=False)
print("Saved detected subscriptions CSV.")

# **OPTIONAL ENHANCEMENTS**
Spell Correction (SymSpell), Lemmatization (NLTK)

In [62]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Samyak.Bora\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Samyak.Bora\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

___
___

**Evaluate Accuracy**

In [ ]:
# Evaluate spaCy model on the TEST SET
correct = 0
total = 0

for text, label in zip(X_test, y_test):
    doc = nlp(text)
    pred = 1 if doc.cats["SUBSCRIPTION"] >= 0.5 else 0
    if pred == label:
        correct += 1
    total += 1

accuracy = correct / total
accuracy

In [ ]:
# Evaluate Accuracy
correct = 0
total = 0

for _, row in df.iterrows():
    doc = nlp(row["CleanedDescription"])
    pred = 1 if doc.cats["SUBSCRIPTION"] >= 0.5 else 0

    if pred == row["SubscriptionFlag"]:
        correct += 1
    total += 1

accuracy = correct / total
accuracy

___
___

In [ ]:
# # Training Loop
# n_iter = 10
# optimizer = nlp.initialize()

# for epoch in range(n_iter):
#     random.shuffle(train_data)
#     losses = {}

#     batches = minibatch(train_data, size=compounding(4.0, 32.0, 1.5))

#     for batch in batches:
#         examples = []
#         for text, annotations in batch:
#             doc = nlp.make_doc(text)
#             examples.append(Example.from_dict(doc, annotations))
#         nlp.update(examples, sgd=optimizer, losses=losses)

#     print(f"Epoch {epoch+1}/{n_iter}, Loss: {losses}")

Epoch 1/10, Loss: {'textcat_multilabel': 0.6193333964600336}
Epoch 2/10, Loss: {'textcat_multilabel': 1.2553283594424203e-06}
Epoch 3/10, Loss: {'textcat_multilabel': 4.0881065122705396e-07}
Epoch 4/10, Loss: {'textcat_multilabel': 1.4279217421669932e-07}
Epoch 5/10, Loss: {'textcat_multilabel': 5.40176219943414e-08}
Epoch 6/10, Loss: {'textcat_multilabel': 2.1644825331482327e-08}
Epoch 7/10, Loss: {'textcat_multilabel': 9.746361631494428e-09}
Epoch 8/10, Loss: {'textcat_multilabel': 4.66132700445215e-09}
Epoch 9/10, Loss: {'textcat_multilabel': 2.3371203418734443e-09}
Epoch 10/10, Loss: {'textcat_multilabel': 1.401930615216665e-09}


In [ ]:
doc = nlp("Monthly Netflix debit processed")
doc.cats

___
___

In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import re
import string
import pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# plt.style.use("seaborn")
pd.set_option("display.max_columns", 50)

# Load cleaned dataset
df = pd.read_csv("..\\data\\subscription_dataset_cleaned.csv")
df.head()

,CustomerID,TransactionID,Date,Description,Merchant,Amount,Balance,SubscriptionFlag,Month,DayOfWeek,TransactionType,Category
0,C1,T1,2025-01-01,Payment to Cafe Coffee Day,Cafe Coffee Day,-1889,6136,0,2025-01,Wednesday,Debit,Food
1,C1,T2,2025-01-01,Payment to Flipkart,Flipkart,-955,5181,0,2025-01,Wednesday,Debit,Shopping
2,C1,T3,2025-01-02,Salary Credit,Employer,49954,55135,0,2025-01,Thursday,Credit,Others
3,C1,T4,2025-01-02,Payment to Reliance Smart,Reliance Smart,-1582,53553,0,2025-01,Thursday,Debit,Shopping
4,C1,T5,2025-01-02,Payment to Petrol Pump,Petrol Pump,-1257,52296,0,2025-01,Thursday,Debit,Fuel


**NLP Text Preprocessing**

In [ ]:
def clean_text(text: str) -> str:
    """Lowercase, remove punctuation, digits, and normalize spaces."""
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.replace("\n", " ")
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["CleanedDescription"] = df["Description"].apply(clean_text)
df.head()

,CustomerID,TransactionID,Date,Description,Merchant,Amount,Balance,SubscriptionFlag,Month,DayOfWeek,TransactionType,Category,CleanedDescription
0,C1,T1,2025-01-01,Payment to Cafe Coffee Day,Cafe Coffee Day,-1889,6136,0,2025-01,Wednesday,Debit,Food,payment to cafe coffee day
1,C1,T2,2025-01-01,Payment to Flipkart,Flipkart,-955,5181,0,2025-01,Wednesday,Debit,Shopping,payment to flipkart
2,C1,T3,2025-01-02,Salary Credit,Employer,49954,55135,0,2025-01,Thursday,Credit,Others,salary credit
3,C1,T4,2025-01-02,Payment to Reliance Smart,Reliance Smart,-1582,53553,0,2025-01,Thursday,Debit,Shopping,payment to reliance smart
4,C1,T5,2025-01-02,Payment to Petrol Pump,Petrol Pump,-1257,52296,0,2025-01,Thursday,Debit,Fuel,payment to petrol pump


**Subscription Keyword Dictionary**

In [ ]:
keyword_dict = {
    "netflix": "video_streaming",
    "amazon prime": "video_streaming",
    "spotify": "music_streaming",
    "gym membership": "fitness",
    "gym": "fitness",
    "electricity bill": "utility",
    "electricity subscription": "utility",
    "subscription": "generic_subscription",
    "membership": "generic_subscription",

    # ----- Future Possible Keywords -----
    # "auto debit": "generic_subscription",
    # "renewal": "generic_subscription",
    # "annual fee": "fee",
    # "monthly fee": "fee",
    # "insurance": "insurance"
}

**Keyword-Based Detector**

In [ ]:
def keyword_detector(text):
    text = clean_text(text)

    for key, cat in keyword_dict.items():
        if key in text:
            return 1, cat, key  # subscription detected

    return 0, None, None


**ML Training Data Prep**

In [ ]:
X_text = df["CleanedDescription"]
y_label = df["SubscriptionFlag"]

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y_label, test_size=0.2, random_state=355
)

X_train.head()


98960        payment to uber
22739       payment to dmart
76614        payment to uber
83774    payment to flipkart
48206      payment to swiggy
Name: CleanedDescription, dtype: object

Train ML Model (TF-IDF + Logistic Regression)

In [ ]:
model_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression())
])

model_pipeline.fit(X_train, y_train)

accuracy = model_pipeline.score(X_test, y_test)
accuracy

1.0

In [ ]:
model_pipeline["tfidf"].get_feature_names_out()[:20]

array(['amazon', 'bill', 'cafe', 'coffee', 'credit', 'day', 'dmart',
       'electricity', 'flipkart', 'gym', 'medical', 'membership',
       'netflix', 'payment', 'petrol', 'prime', 'pump', 'reliance',
       'salary', 'smart'], dtype=object)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     18619
           1       1.00      1.00      1.00      1221

    accuracy                           1.00     19840
   macro avg       1.00      1.00      1.00     19840
weighted avg       1.00      1.00      1.00     19840

